In [2]:

from pathlib import Path
import subprocess
import sys

repo = Path("/content/ulba-flood-forecasting")

if repo.exists():
    subprocess.run(
        ["git", "-C", str(repo), "pull", "origin", "main"],
        check=True
    )
else:
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/lysa106/ulba-flood-forecasting.git",
            str(repo)
        ],
        check=True
    )

sys.path.insert(0, str(repo))

from src.preprocessing import (
    load_data,
    check_data,
    prepare_data
)

print("Preprocessing module imported successfully!")



Preprocessing module imported successfully!


In [3]:

from google.colab import files

uploaded = files.upload()


Saving Ulba_combined_PROVISIONAL_gauge_IMERG_MODIS_2013_2023.csv to Ulba_combined_PROVISIONAL_gauge_IMERG_MODIS_2013_2023.csv


In [4]:

from pathlib import Path

csv_files = list(Path("/content").glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Upload the Ulba CSV first.")

df = load_data(csv_files[0])
check_data(df)

print("\nAvailable columns:")
print(df.columns.tolist())


Dataset shape: (4017, 10)
Columns: ['date', 'station_code', 'discharge_m3_s', 'precipitation_mm', 'observations', 'snow_cover_mean', 'valid_pixel_count', 'image_count', 'snow_data_missing', 'spatial_coverage']

First five rows:
         date  station_code  discharge_m3_s  precipitation_mm  observations  \
0  2013-01-01         11164            22.7          0.000000            48   
1  2013-01-02         11164            23.1          0.063317            48   
2  2013-01-03         11164            23.5          0.000000            48   
3  2013-01-04         11164            23.4          0.031387            48   
4  2013-01-05         11164            23.3          0.051083            48   

   snow_cover_mean  valid_pixel_count  image_count  snow_data_missing  \
0        65.721199                147            1                  0   
1        54.878951               1117            1                  0   
2        53.333181               1963            1                  0   
3    

In [5]:

# Step 5: Prepare and split the dataset

df = prepare_data(df, date_column="date")

# Chronological train / validation / test split
train = df[
    (df["date"] >= "2013-01-01") &
    (df["date"] <= "2019-12-31")
].copy()

val = df[
    (df["date"] >= "2020-01-01") &
    (df["date"] <= "2021-12-31")
].copy()

test = df[
    (df["date"] >= "2022-01-01") &
    (df["date"] <= "2023-12-31")
].copy()

print("\nDATASET SPLIT")
print("Training:", len(train), "days")
print("Validation:", len(val), "days")
print("Testing:", len(test), "days")

print("\nMissing values in training:")
print(train.isna().sum())

print("\nMissing values in validation:")
print(val.isna().sum())

print("\nMissing values in testing:")
print(test.isna().sum())



Start date: 2013-01-01 00:00:00
End date: 2023-12-31 00:00:00
Total observations: 4017

DATASET SPLIT
Training: 2556 days
Validation: 731 days
Testing: 730 days

Missing values in training:
date                   0
station_code           0
discharge_m3_s         0
precipitation_mm       0
observations           0
snow_cover_mean      413
valid_pixel_count      0
image_count            0
snow_data_missing      0
spatial_coverage       0
dtype: int64

Missing values in validation:
date                   0
station_code           0
discharge_m3_s         0
precipitation_mm       0
observations           0
snow_cover_mean      116
valid_pixel_count      0
image_count            0
snow_data_missing      0
spatial_coverage       0
dtype: int64

Missing values in testing:
date                  0
station_code          0
discharge_m3_s        0
precipitation_mm      0
observations          0
snow_cover_mean      99
valid_pixel_count     0
image_count           0
snow_data_missing     0
spatial_

In [6]:

import numpy as np
from sklearn.preprocessing import StandardScaler

# LSTM A: gauge discharge only
FEATURES = ["discharge_m3_s"]
TARGET = "discharge_m3_s"
LOOKBACK = 14

# Fit scalers on training data only
x_scaler = StandardScaler()
y_scaler = StandardScaler()

x_scaler.fit(train[FEATURES])
y_scaler.fit(train[[TARGET]])

# Transform each period
train_x = x_scaler.transform(train[FEATURES])
val_x = x_scaler.transform(val[FEATURES])
test_x = x_scaler.transform(test[FEATURES])

train_y = y_scaler.transform(train[[TARGET]])
val_y = y_scaler.transform(val[[TARGET]])
test_y = y_scaler.transform(test[[TARGET]])

# Create 14-day input sequences
def create_sequences(X, y, lookback=14):
    X_seq, y_seq = [], []

    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])

    return np.array(X_seq), np.array(y_seq)

X_train, y_train = create_sequences(train_x, train_y, LOOKBACK)
X_val, y_val = create_sequences(val_x, val_y, LOOKBACK)
X_test, y_test = create_sequences(test_x, test_y, LOOKBACK)

print("LSTM A sequences:")
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)


LSTM A sequences:
Training: (2542, 14, 1) (2542, 1)
Validation: (717, 14, 1) (717, 1)
Testing: (716, 14, 1) (716, 1)


In [7]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import copy
import random

# Reproducibility
SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Model configuration
HIDDEN_SIZE = 32
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 100
PATIENCE = 10

# Convert data into PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Define LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_output = output[:, -1, :]
        return self.fc(last_output)

model_a = LSTMModel(
    input_size=1,
    hidden_size=HIDDEN_SIZE
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model_a.parameters(),
    lr=LEARNING_RATE
)

# Training with early stopping
best_val_loss = float("inf")
best_weights = None
patience_counter = 0

for epoch in range(EPOCHS):
    model_a.train()
    train_losses = []

    for xb, yb in train_loader:
        optimizer.zero_grad()

        predictions = model_a(xb)
        loss = criterion(predictions, yb)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model_a.eval()

    with torch.no_grad():
        val_predictions = model_a(X_val_t)
        val_loss = criterion(
            val_predictions,
            y_val_t
        ).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model_a.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}: "
            f"Train Loss = {np.mean(train_losses):.6f}, "
            f"Val Loss = {val_loss:.6f}"
        )

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Restore best validation model
model_a.load_state_dict(best_weights)

print("\nLSTM A training completed!")


Epoch 10: Train Loss = 0.059987, Val Loss = 0.021148
Epoch 20: Train Loss = 0.051610, Val Loss = 0.018066
Epoch 30: Train Loss = 0.046784, Val Loss = 0.018983
Early stopping at epoch 32

LSTM A training completed!


In [8]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict on test data
model_a.eval()

with torch.no_grad():
    predictions_scaled = model_a(
        torch.tensor(X_test, dtype=torch.float32)
    ).numpy()

# Convert back to original discharge units (m3/s)
y_pred = y_scaler.inverse_transform(
    predictions_scaled
).flatten()

y_true = y_scaler.inverse_transform(
    y_test
).flatten()

# Evaluation metrics
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    nse = 1 - np.sum((y_true - y_pred) ** 2) / denominator

    r = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = np.std(y_pred) / np.std(y_true)
    beta = np.mean(y_pred) / np.mean(y_true)

    kge = 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )

    # Positive PBIAS means underprediction
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "NSE": nse,
        "KGE": kge,
        "PBIAS (%)": pbias
    }

metrics_a = calculate_metrics(y_true, y_pred)

print("LSTM A — TEST RESULTS (2022–2023)")
print("-" * 40)

for metric, value in metrics_a.items():
    print(f"{metric}: {value:.4f}")

# Save test predictions with their target dates
prediction_dates = test["date"].iloc[LOOKBACK:].reset_index(drop=True)

results_a = pd.DataFrame({
    "date": prediction_dates,
    "observed_discharge": y_true,
    "predicted_discharge": y_pred
})

print("\nFirst five predictions:")
print(results_a.head())


LSTM A — TEST RESULTS (2022–2023)
----------------------------------------
RMSE: 29.2457
MAE: 10.4304
NSE: 0.9419
KGE: 0.9551
PBIAS (%): 0.4067

First five predictions:
        date  observed_discharge  predicted_discharge
0 2022-01-15                15.3            15.868953
1 2022-01-16                15.3            15.868953
2 2022-01-17                15.3            15.868953
3 2022-01-18                15.3            15.868953
4 2022-01-19                15.3            15.868953


In [9]:

# Step 9: Persistence Baseline

# Use the last observed discharge from each 14-day window
persistence_pred = x_scaler.inverse_transform(
    X_test[:, -1, :]
).flatten()

# Evaluate on exactly the same dates as LSTM A
metrics_persistence = calculate_metrics(
    y_true,
    persistence_pred
)

# Compare models
comparison = pd.DataFrame({
    "LSTM A": metrics_a,
    "Persistence": metrics_persistence
}).T

print("MODEL COMPARISON")
print(comparison.round(4))

# RMSE improvement
improvement = (
    (metrics_persistence["RMSE"] - metrics_a["RMSE"])
    / metrics_persistence["RMSE"]
) * 100

print(f"\nLSTM A RMSE improvement: {improvement:.2f}%")



MODEL COMPARISON
                RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A       29.2457  10.4304  0.9419  0.9551     0.4067
Persistence  34.3700  11.9494  0.9198  0.9599     0.0014

LSTM A RMSE improvement: 14.91%


In [10]:

# Step 10: Seasonal Climatology Baseline

import pandas as pd
import numpy as np

# Day of year represented as month-day
train_clim = train.copy()
train_clim["month_day"] = train_clim["date"].dt.strftime("%m-%d")

# Compute climatology using training data only
daily_climatology = train_clim.groupby("month_day")[
    "discharge_m3_s"
].mean()

# Match the same 716 target dates used by LSTM A
test_dates = test["date"].iloc[LOOKBACK:].reset_index(drop=True)
month_days = test_dates.dt.strftime("%m-%d")

climatology_pred = month_days.map(daily_climatology)

# Fallback for any missing calendar days
climatology_pred = climatology_pred.fillna(
    train["discharge_m3_s"].mean()
).to_numpy()

# Evaluate climatology
metrics_climatology = calculate_metrics(
    y_true,
    climatology_pred
)

# Compare all three models
comparison = pd.DataFrame({
    "LSTM A": metrics_a,
    "Persistence": metrics_persistence,
    "Seasonal Climatology": metrics_climatology
}).T

print("MODEL COMPARISON: LSTM A vs BASELINES")
print(comparison.round(4))

# Identify strongest baseline based on RMSE
baseline_rmse = {
    "Persistence": metrics_persistence["RMSE"],
    "Seasonal Climatology": metrics_climatology["RMSE"]
}

best_baseline = min(baseline_rmse, key=baseline_rmse.get)

print("\nBest baseline:", best_baseline)
print("Baseline RMSE:", round(baseline_rmse[best_baseline], 4))


MODEL COMPARISON: LSTM A vs BASELINES
                         RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A                29.2457  10.4304  0.9419  0.9551     0.4067
Persistence           34.3700  11.9494  0.9198  0.9599     0.0014
Seasonal Climatology  80.2486  48.0755  0.5629  0.6286   -28.1763

Best baseline: Persistence
Baseline RMSE: 34.37


In [11]:

# Step 11: Prepare gridded data for LSTM B and C

import numpy as np
from sklearn.preprocessing import StandardScaler

GRID_FEATURES = [
    "precipitation_mm",
    "snow_cover_mean"
]

COMBINED_FEATURES = [
    "discharge_m3_s",
    "precipitation_mm",
    "snow_cover_mean"
]

# Work with a chronological copy of the full dataset
data_bc = df.copy().sort_values("date").reset_index(drop=True)

# Fill only short gaps using past observations
data_bc["snow_cover_mean"] = (
    data_bc["snow_cover_mean"].ffill(limit=2)
)

# The same target dates must be used for both models
def create_valid_sequences(data, features, lookback=14):
    X, y, dates = [], [], []

    for i in range(lookback, len(data)):
        window = data.iloc[i-lookback:i][features]
        target = data.iloc[i]["discharge_m3_s"]

        if window.isna().any().any() or pd.isna(target):
            continue

        X.append(window.to_numpy(dtype=float))
        y.append(target)
        dates.append(data.iloc[i]["date"])

    return (
        np.array(X),
        np.array(y),
        pd.to_datetime(dates)
    )

X_b_raw, y_b_raw, dates_b = create_valid_sequences(
    data_bc, GRID_FEATURES
)

X_c_raw, y_c_raw, dates_c = create_valid_sequences(
    data_bc, COMBINED_FEATURES
)

print("LSTM B sequences:", X_b_raw.shape)
print("LSTM C sequences:", X_c_raw.shape)

print("\nSame target dates:",
      np.array_equal(dates_b, dates_c))

print("Total valid target dates:", len(dates_b))

# Check chronological distribution
for name, start, end in [
    ("Train", "2013-01-01", "2019-12-31"),
    ("Validation", "2020-01-01", "2021-12-31"),
    ("Test", "2022-01-01", "2023-12-31")
]:
    count = ((dates_b >= start) & (dates_b <= end)).sum()
    print(f"{name}: {count} valid targets")


LSTM B sequences: (3566, 14, 2)
LSTM C sequences: (3566, 14, 3)

Same target dates: True
Total valid target dates: 3566
Train: 2244 valid targets
Validation: 661 valid targets
Test: 661 valid targets


In [12]:

# Step 12: Scale and split LSTM B and C data

import numpy as np
from sklearn.preprocessing import StandardScaler

def prepare_model_data(X_raw, y_raw, dates):
    # Split by target date
    train_mask = dates < "2020-01-01"
    val_mask = (dates >= "2020-01-01") & (dates < "2022-01-01")
    test_mask = dates >= "2022-01-01"

    # Fit scalers on training data only
    n_features = X_raw.shape[2]

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    X_train_raw = X_raw[train_mask]
    y_train_raw = y_raw[train_mask]

    scaler_x.fit(
        X_train_raw.reshape(-1, n_features)
    )
    scaler_y.fit(
        y_train_raw.reshape(-1, 1)
    )

    # Transform sequences
    def transform_X(X):
        original_shape = X.shape
        return scaler_x.transform(
            X.reshape(-1, n_features)
        ).reshape(original_shape).astype(np.float32)

    def transform_y(y):
        return scaler_y.transform(
            y.reshape(-1, 1)
        ).astype(np.float32)

    return {
        "X_train": transform_X(X_raw[train_mask]),
        "y_train": transform_y(y_raw[train_mask]),
        "X_val": transform_X(X_raw[val_mask]),
        "y_val": transform_y(y_raw[val_mask]),
        "X_test": transform_X(X_raw[test_mask]),
        "y_test": transform_y(y_raw[test_mask]),
        "test_dates": dates[test_mask],
        "scaler_x": scaler_x,
        "scaler_y": scaler_y
    }


data_b = prepare_model_data(
    X_b_raw, y_b_raw, dates_b
)

data_c = prepare_model_data(
    X_c_raw, y_c_raw, dates_c
)

print("LSTM B:")
print("Train:", data_b["X_train"].shape)
print("Validation:", data_b["X_val"].shape)
print("Test:", data_b["X_test"].shape)

print("\nLSTM C:")
print("Train:", data_c["X_train"].shape)
print("Validation:", data_c["X_val"].shape)
print("Test:", data_c["X_test"].shape)

print("\nSame test dates:",
      np.array_equal(
          data_b["test_dates"],
          data_c["test_dates"]
      ))


LSTM B:
Train: (2244, 14, 2)
Validation: (661, 14, 2)
Test: (661, 14, 2)

LSTM C:
Train: (2244, 14, 3)
Validation: (661, 14, 3)
Test: (661, 14, 3)

Same test dates: True


In [13]:

# Step 13: Train LSTM B and LSTM C

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import random
import copy

def train_lstm(data, model_name, seed=17):
    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    X_train_t = torch.tensor(
        data["X_train"], dtype=torch.float32
    )
    y_train_t = torch.tensor(
        data["y_train"], dtype=torch.float32
    )

    X_val_t = torch.tensor(
        data["X_val"], dtype=torch.float32
    )
    y_val_t = torch.tensor(
        data["y_val"], dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=32,
        shuffle=True
    )

    model = LSTMModel(
        input_size=data["X_train"].shape[2],
        hidden_size=32
    )

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.001
    )
    criterion = nn.MSELoss()

    best_loss = float("inf")
    best_weights = None
    patience_counter = 0

    print(f"\nTraining {model_name}...")

    for epoch in range(100):
        model.train()
        losses = []

        for xb, yb in loader:
            optimizer.zero_grad()
            predictions = model(xb)
            loss = criterion(predictions, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()

        with torch.no_grad():
            val_loss = criterion(
                model(X_val_t), y_val_t
            ).item()

        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(
                model.state_dict()
            )
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}: "
                f"Train={np.mean(losses):.5f}, "
                f"Val={val_loss:.5f}"
            )

        if patience_counter >= 10:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_weights)
    model.eval()

    print(f"{model_name} training completed!")

    return model


# Train both models
model_b = train_lstm(data_b, "LSTM B")
model_c = train_lstm(data_c, "LSTM C")

print("\nBoth models trained successfully!")



Training LSTM B...
Epoch 10: Train=0.57744, Val=0.81522
Early stopping at epoch 11
LSTM B training completed!

Training LSTM C...
Epoch 10: Train=0.04232, Val=0.03384
Epoch 20: Train=0.02877, Val=0.02559
Epoch 30: Train=0.02405, Val=0.02382
Epoch 40: Train=0.02114, Val=0.02662
Early stopping at epoch 46
LSTM C training completed!

Both models trained successfully!


In [15]:


import torch
import numpy as np

def predict_lstm(model, data):
    model.eval()

    X_test_t = torch.tensor(
        data["X_test"],
        dtype=torch.float32
    )

    with torch.no_grad():
        predictions_scaled = model(X_test_t).cpu().numpy()

    y_pred = data["scaler_y"].inverse_transform(
        predictions_scaled
    ).flatten()

    y_true = data["scaler_y"].inverse_transform(
        data["y_test"]
    ).flatten()

    return y_true, y_pred

print("predict_lstm function is ready!")


predict_lstm function is ready!


In [16]:
import numpy as np
import pandas as pd

# Predictions from LSTM B and C
y_b_true, y_b_pred = predict_lstm(model_b, data_b)
y_c_true, y_c_pred = predict_lstm(model_c, data_c)

# Build prediction table
comparison_df = pd.DataFrame({
    "date": pd.to_datetime(data_c["test_dates"]),
    "observed": y_c_true,
    "LSTM B": y_b_pred,
    "LSTM C": y_c_pred
})

# Add LSTM A predictions by date
a_df = results_a.copy()
a_df["date"] = pd.to_datetime(a_df["date"])

a_df = a_df.rename(columns={
    "predicted_discharge": "LSTM A"
})

comparison_df = comparison_df.merge(
    a_df[["date", "LSTM A"]],
    on="date",
    how="inner",
    validate="one_to_one"
)

# Persistence: previous day's observed discharge
previous_discharge = df[
    ["date", "discharge_m3_s"]
].copy()

previous_discharge["date"] = (
    previous_discharge["date"] + pd.Timedelta(days=1)
)

previous_discharge = previous_discharge.rename(
    columns={"discharge_m3_s": "Persistence"}
)

comparison_df = comparison_df.merge(
    previous_discharge,
    on="date",
    how="left",
    validate="one_to_one"
)

# Seasonal climatology based on training data only
comparison_df["month_day"] = (
    comparison_df["date"].dt.strftime("%m-%d")
)

comparison_df["Seasonal Climatology"] = (
    comparison_df["month_day"]
    .map(daily_climatology)
    .fillna(train["discharge_m3_s"].mean())
)

# Ensure all predictions are available
model_names = [
    "LSTM A",
    "LSTM B",
    "LSTM C",
    "Persistence",
    "Seasonal Climatology"
]

comparison_df = comparison_df.dropna(
    subset=["observed"] + model_names
).sort_values("date").reset_index(drop=True)

if comparison_df.empty:
    raise ValueError("No common test dates found.")

# Check that B and C use the same observations
b_observed = pd.Series(
    y_b_true,
    index=pd.to_datetime(data_b["test_dates"])
)
aligned_b = b_observed.loc[comparison_df["date"]].to_numpy()

assert np.allclose(
    comparison_df["observed"].to_numpy(),
    aligned_b
)

# Calculate metrics on identical dates
y_common = comparison_df["observed"].to_numpy()

all_metrics = {
    name: calculate_metrics(
        y_common,
        comparison_df[name].to_numpy()
    )
    for name in model_names
}

comparison_all = pd.DataFrame(all_metrics).T

print("FAIR PILOT MODEL COMPARISON")
print("Common test dates:", len(comparison_df))
print(comparison_all.round(4))

rmse_c = all_metrics["LSTM C"]["RMSE"]
rmse_p = all_metrics["Persistence"]["RMSE"]

improvement_c = 100 * (rmse_p - rmse_c) / rmse_p

print(
    f"\nLSTM C improvement over Persistence: "
    f"{improvement_c:.2f}%"
)


FAIR PILOT MODEL COMPARISON
Common test dates: 647
                          RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A                 29.2326  10.6715  0.9447  0.9528     0.8635
LSTM B                120.7077  83.4803  0.0568  0.0495   -16.8855
LSTM C                 25.8860  10.8971  0.9566  0.9176     3.5376
Persistence            33.7565  12.0253  0.9262  0.9621     0.6357
Seasonal Climatology   80.2435  47.9736  0.5832  0.6462   -27.3562

LSTM C improvement over Persistence: 23.32%


In [17]:


import numpy as np
import pandas as pd

SEEDS = [17, 29, 41]

ABLATION_CONFIGS = {
    "All predictors": [
        "discharge_m3_s",
        "precipitation_mm",
        "snow_cover_mean"
    ],
    "Without precipitation": [
        "discharge_m3_s",
        "snow_cover_mean"
    ],
    "Without snow cover": [
        "discharge_m3_s",
        "precipitation_mm"
    ]
}

# Use exactly the same target dates for all configurations
common_ablation_dates = pd.DatetimeIndex(
    comparison_df["date"]
)

ablation_results = []

for config_name, features in ABLATION_CONFIGS.items():

    print(f"\nConfiguration: {config_name}")

    X_raw, y_raw, dates = create_valid_sequences(
        data_bc,
        features,
        lookback=LOOKBACK
    )

    prepared = prepare_model_data(
        X_raw,
        y_raw,
        dates
    )

    run_rmse = []

    for seed in SEEDS:
        model = train_lstm(
            prepared,
            f"{config_name} (seed={seed})",
            seed=seed
        )

        y_actual, y_predicted = predict_lstm(
            model,
            prepared
        )

        # Align predictions with the common test dates
        pred_series = pd.Series(
            y_predicted,
            index=pd.DatetimeIndex(prepared["test_dates"])
        )

        actual_series = pd.Series(
            y_actual,
            index=pd.DatetimeIndex(prepared["test_dates"])
        )

        aligned_pred = pred_series.reindex(
            common_ablation_dates
        )
        aligned_actual = actual_series.reindex(
            common_ablation_dates
        )

        if aligned_pred.isna().any():
            raise ValueError(
                f"Missing predictions for {config_name}"
            )

        rmse = calculate_metrics(
            aligned_actual.to_numpy(),
            aligned_pred.to_numpy()
        )["RMSE"]

        run_rmse.append(rmse)

        print(f"Seed {seed}: RMSE = {rmse:.4f}")

    ablation_results.append({
        "Configuration": config_name,
        "RMSE mean": np.mean(run_rmse),
        "RMSE SD": np.std(run_rmse, ddof=1)
    })

ablation_table = pd.DataFrame(ablation_results)

baseline_rmse = ablation_table.loc[
    ablation_table["Configuration"] == "All predictors",
    "RMSE mean"
].iloc[0]

ablation_table["RMSE change"] = (
    ablation_table["RMSE mean"] - baseline_rmse
)

print("\nABLATION STUDY RESULTS")
print(ablation_table.round(4).to_string(index=False))



Configuration: All predictors

Training All predictors (seed=17)...
Epoch 10: Train=0.04232, Val=0.03384
Epoch 20: Train=0.02877, Val=0.02559
Epoch 30: Train=0.02405, Val=0.02382
Epoch 40: Train=0.02114, Val=0.02662
Early stopping at epoch 46
All predictors (seed=17) training completed!
Seed 17: RMSE = 25.8860

Training All predictors (seed=29)...
Epoch 10: Train=0.04612, Val=0.03509
Epoch 20: Train=0.02962, Val=0.02803
Epoch 30: Train=0.02423, Val=0.02402
Epoch 40: Train=0.02023, Val=0.02308
Early stopping at epoch 41
All predictors (seed=29) training completed!
Seed 29: RMSE = 25.9674

Training All predictors (seed=41)...
Epoch 10: Train=0.04410, Val=0.04143
Epoch 20: Train=0.03139, Val=0.02944
Epoch 30: Train=0.02407, Val=0.02939
Early stopping at epoch 39
All predictors (seed=41) training completed!
Seed 41: RMSE = 26.6998

Configuration: Without precipitation

Training Without precipitation (seed=17)...
Epoch 10: Train=0.05960, Val=0.03152
Epoch 20: Train=0.04171, Val=0.01905
Epo

In [18]:

# Step 16: Statistical Testing
# Moving Block Bootstrap + Paired Block Sign-Flip Test

import numpy as np
import pandas as pd

N_BOOTSTRAP = 1000
BLOCK_SIZE = 14
SEED = 42

def rmse(y, pred):
    return np.sqrt(np.mean((y - pred) ** 2))


def rmse_reduction(y, pred_reference, pred_model):
    reference_rmse = rmse(y, pred_reference)
    model_rmse = rmse(y, pred_model)

    return 100 * (
        reference_rmse - model_rmse
    ) / reference_rmse


def statistical_comparison(
    y, reference, model, n_bootstrap=1000,
    block_size=14, seed=42
):
    rng = np.random.default_rng(seed)
    n = len(y)

    observed_reduction = rmse_reduction(
        y, reference, model
    )

    # Moving block bootstrap
    bootstrap_reductions = []

    for _ in range(n_bootstrap):
        indices = []

        while len(indices) < n:
            start = rng.integers(
                0, max(1, n - block_size + 1)
            )
            indices.extend(
                range(start, min(start + block_size, n))
            )

        indices = np.array(indices[:n])

        reduction = rmse_reduction(
            y[indices],
            reference[indices],
            model[indices]
        )

        bootstrap_reductions.append(reduction)

    ci_low, ci_high = np.percentile(
        bootstrap_reductions, [2.5, 97.5]
    )

    # Paired block sign-flip test on squared error differences
    loss_difference = (
        (y - reference) ** 2
        - (y - model) ** 2
    )

    observed_stat = np.abs(
        np.mean(loss_difference)
    )

    blocks = [
        loss_difference[i:i + block_size]
        for i in range(0, n, block_size)
    ]

    extreme_count = 0

    for _ in range(n_bootstrap):
        flipped = np.concatenate([
            block * rng.choice([-1, 1])
            for block in blocks
        ])

        test_stat = np.abs(np.mean(flipped))

        if test_stat >= observed_stat:
            extreme_count += 1

    p_value = (
        extreme_count + 1
    ) / (n_bootstrap + 1)

    return {
        "RMSE reduction (%)": observed_reduction,
        "CI lower (%)": ci_low,
        "CI upper (%)": ci_high,
        "p-value": p_value
    }


# Use the matched comparison dataset
y = comparison_df["observed"].to_numpy()

comparisons = {
    "C vs Persistence": "Persistence",
    "C vs LSTM A": "LSTM A",
    "C vs LSTM B": "LSTM B"
}

stat_results = []

for label, reference_name in comparisons.items():
    result = statistical_comparison(
        y=y,
        reference=comparison_df[
            reference_name
        ].to_numpy(),
        model=comparison_df[
            "LSTM C"
        ].to_numpy(),
        n_bootstrap=N_BOOTSTRAP,
        block_size=BLOCK_SIZE,
        seed=SEED
    )

    result["Comparison"] = label
    stat_results.append(result)

stats_table = pd.DataFrame(stat_results)

print("STATISTICAL TEST RESULTS")
print(stats_table.round(4).to_string(index=False))


STATISTICAL TEST RESULTS
 RMSE reduction (%)  CI lower (%)  CI upper (%)  p-value       Comparison
            23.3154       10.4533       31.5156   0.0500 C vs Persistence
            11.4482       -2.5182       21.7613   0.1568      C vs LSTM A
            78.5548       74.9711       83.1474   0.0010      C vs LSTM B


In [19]:

# Step 17: Save experimental results

import os
import pandas as pd
from google.colab import files

# Create results folder
os.makedirs("results", exist_ok=True)

# Save model comparison
comparison_all.to_csv(
    "results/model_comparison.csv",
    index_label="Model"
)

# Save predictions
comparison_df.to_csv(
    "results/test_predictions.csv",
    index=False
)

# Save ablation study
if "ablation_table" in globals():
    ablation_table.to_csv(
        "results/ablation_results.csv",
        index=False
    )

# Save statistical tests
if "stats_table" in globals():
    stats_table.to_csv(
        "results/statistical_tests.csv",
        index=False
    )

print("Results saved successfully!")

print("\nSaved files:")
print(os.listdir("results"))


Results saved successfully!

Saved files:
['test_predictions.csv', 'ablation_results.csv', 'figures', 'model_comparison.csv', 'statistical_tests.csv']


In [20]:

# Step 18: Download results

import shutil
from google.colab import files

shutil.make_archive(
    "ulba_pilot_results",
    "zip",
    "results"
)

files.download("ulba_pilot_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:

# Final check of all experiments

required = [
    "model_a",
    "model_b",
    "model_c",
    "comparison_all",
    "ablation_table",
    "stats_table",
    "comparison_df"
]

missing = [x for x in required if x not in globals()]

if missing:
    print("Missing:", missing)
else:
    print("ALL EXPERIMENTS COMPLETED SUCCESSFULLY!")
    print("\nModel comparison:")
    print(comparison_all.round(4))

    print("\nAblation study:")
    print(ablation_table.round(4))

    print("\nStatistical tests:")
    print(stats_table.round(4))


ALL EXPERIMENTS COMPLETED SUCCESSFULLY!

Model comparison:
                          RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A                 29.2326  10.6715  0.9447  0.9528     0.8635
LSTM B                120.7077  83.4803  0.0568  0.0495   -16.8855
LSTM C                 25.8860  10.8971  0.9566  0.9176     3.5376
Persistence            33.7565  12.0253  0.9262  0.9621     0.6357
Seasonal Climatology   80.2435  47.9736  0.5832  0.6462   -27.3562

Ablation study:
           Configuration  RMSE mean  RMSE SD  RMSE change
0         All predictors    26.1844   0.4482       0.0000
1  Without precipitation    29.9781   0.5428       3.7937
2     Without snow cover    25.0055   0.8953      -1.1789

Statistical tests:
   RMSE reduction (%)  CI lower (%)  CI upper (%)  p-value        Comparison
0             23.3154       10.4533       31.5156   0.0500  C vs Persistence
1             11.4482       -2.5182       21.7613   0.1568       C vs LSTM A
2             78.5548       74.9711    